In [1]:
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, and reporting of experiment
# results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `lmetk_report` package.
import sys
import pandas as pd
sys.path.append('..')

from lmetk_report.run_processor import load_runs_df
from lmetk_report.analysis import analyze_results
from lmetk_report.reporting import create_summary_table

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "TabPHM-autoencoders-checks_06_10_night"
USER = "imos-buildingenergy-1"
METRIC_PREFIXES = ["val/", "test/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse"]

# Configure pandas display options for better readability in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
df, config_columns, dropped_columns = load_runs_df(
    project_name=PROJECT_NAME, 
    user=USER
)

# ---
# ## 4. Core Analysis
# Run the main analysis function to group results by dataset and model,
# find the best hyperparameter configuration, and collect performance metrics.
# ---
all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES
)

# ---
# ## 5. Generate and Display Summary
# Create the final pivoted summary table for a high-level overview
# of the best performance for each model on each dataset.
# ---
summary_df = create_summary_table(all_results)

print("\n📊 Final Performance Summary Table:")
display(summary_df)



wandb: Currently logged in as: levsap to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Fetching runs from wandb project: imos-buildingenergy-1/TabPHM-autoencoders-checks_06_10_night...
Skipping run 'xjtu_sy+reconstruction+residual_autoencoder_dd53c58d' with state 'running'
Skipping run 'xjtu_sy+reconstruction+residual_autoencoder_3f75d911' with state 'running'
Saved 6 runs to cache: csv_files/TabPHM-autoencoders-checks_06_10_night.csv
Dropped 10 columns with >90% NaN values.
Dropped 12 columns based on name filters (e.g., 'paths.').
Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21 is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21:
model.kernels: [[5, 5, 5, 5, 5], [7, 7, 5, 5, 5], [10, 7, 7, 7, 7, 7, 7, 7, 7, 7]]
model.strides: [[1, 1, 1, 1, 1], [2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
model.dilations: [[1, 2, 4, 8, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
model.output_channels: [[16, 32, 64, 64, 64], [16, 32, 64, 64, 64, 16, 32, 64, 64, 64]]
---------
Optimized met

Dataset                                                      PRONOSTIA                                                                                                                          \
Metric                                                       test/loss            test/mae            test/mse           test/rmse            val/loss             val/mae             val/mse   
Model                                                                                                                                                                                            
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0000584526 ± nan  0.0049781874 ± nan  0.0000584526 ± nan  0.0076454296 ± nan  0.0000584526 ± nan  0.0049781874 ± nan  0.0000584526 ± nan   

Dataset                                                                            UNIBO21                                                                                                      \
Metric                                                        val/rmse           test/loss            test/mae            test/mse           test/rmse            val/loss             val/mae   
Model                                                                                                                                                                                            
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0076454296 ± nan  0.0000211729 ± nan  0.0039780731 ± nan  0.0000211729 ± nan  0.0046013985 ± nan  0.0000211729 ± nan  0.0039780731 ± nan   

Dataset                                                                                     
Metric                                                         val/mse            val/rmse  
Model                                                                                       
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0000211729 ± nan  0.0046013985 ± nan

In [2]:
for key in all_results.keys():
    print(f"Dataset key: {key}")
    for key_2 in all_results[key].keys():
        print(f"  Model key: {key_2}")
        for sub_key in all_results[key][key_2].keys():
            #if pd data frame then display the head only
            if isinstance(all_results[key][key_2][sub_key], pd.DataFrame):
                print(f"    {sub_key}:")
                display(all_results[key][key_2][sub_key].head(2))
            # if dict proceed with keys 
            elif isinstance(all_results[key][key_2][sub_key], dict):
                # same for inside dict and repeat
                for inner_key in all_results[key][key_2][sub_key].keys():
                    # print the 
                    if isinstance(all_results[key][key_2][sub_key][inner_key], pd.DataFrame):
                        print(f"   {sub_key}   {inner_key}:")
                        display(all_results[key][key_2][sub_key][inner_key].head(2))
                    else:
                        print(f" {sub_key}     {inner_key}: {all_results[key][key_2][sub_key][inner_key]}")
                
            else:
                print(f"    {sub_key}: {all_results[key][key_2][sub_key]}")
            
            


Dataset key: UNIBO21
  Model key: lmetk.model.autoencoder.wrapper.AutoencoderWrapper
 best_hyperparameters     model.kernels: (5, 5, 5, 5, 5)
 best_hyperparameters     model.strides: (1, 1, 1, 1, 1)
 best_hyperparameters     model.dilations: (1, 2, 4, 8, 1)
 best_hyperparameters     model.output_channels: (16, 32, 64, 64, 64)
 best_hyperparameters     run_names: ['unibo+reconstruction+residual_autoencoder_85baa252']
 best_performance     optimized_on: {'metric': 'val/loss', 'strategy': 'min'}
 best_performance     metrics: defaultdict(<class 'dict'>, {'loss': {'val': {'mean': np.float64(2.1172860215301625e-05), 'std': np.float64(nan)}, 'test': {'mean': np.float64(2.1172860215301625e-05), 'std': np.float64(nan)}}, 'mse': {'val': {'mean': np.float64(2.117286749125924e-05), 'std': np.float64(nan)}, 'test': {'mean': np.float64(2.117286749125924e-05), 'std': np.float64(nan)}}, 'mae': {'val': {'mean': np.float64(0.003978073131293058), 'std': np.float64(nan)}, 'test': {'mean': np.float64(0.00

Model                    model.kernels                   model.strides                 model.dilations                     model.output_channels  \
                                                                                                                                                                                                  
0  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (5, 5, 5, 5, 5)                 (1, 1, 1, 1, 1)                 (1, 2, 4, 8, 1)                      (16, 32, 64, 64, 64)   
2  lmetk.model.autoencoder.wrapper.AutoencoderWra...  (10, 7, 7, 7, 7, 7, 7, 7, 7, 7)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (16, 32, 64, 64, 64, 16, 32, 64, 64, 64)   

   val/loss       val/mse       val/mae      val/rmse     test/loss      test/mse      test/mae     test/rmse                                               run_name  
       mean std      mean std      mean std      mean std      mean std      mean std      mean std      mean std                                               list  
0  0.000021 NaN  0.000021 NaN  0.003978 NaN  0.004601 NaN  0.000021 NaN  0.000021 NaN  0.003978 NaN  0.004601 NaN  [unibo+reconstruction+residual_autoencoder_85b...  
2  0.000061 NaN  0.000061 NaN  0.006618 NaN  0.007828 NaN  0.000061 NaN  0.000061 NaN  0.006618 NaN  0.007828 NaN  [unibo+reconstruction+residual_autoencoder_51f...

    non_aggregated_df:


,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,_runtime,_step,_timestamp,_wandb,epoch,lr-Adam,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/loss_step,train/mae,train/mse,train/rmse,trainer/global_step,val/loss,val/mae,val/mse,val/rmse,loss,seed,tags,test,cache,train,logger,trainer,exp_name,evaluator,hydra_cfg,pin_memory,transforms,num_threads,num_workers,shuffle_val,shuffle_test,shuffle_train,val_batch_size,prefetch_factor,test_batch_size,experiment_group,train_batch_size,use_batch_sampler,model/params/total,model/params/trainable,run_name,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.fold,datasource.cache_dir,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.requires_training,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,model.output_channels,model.blocks_config,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,datamodule._target_,datamodule.num_workers,datamodule.shuffle_val,datamodule.shuffle_test,datamodule.shuffle_train,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.check_finite,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint.verbose,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.save_last,callbacks.model_checkpoint.save_top_k,callbacks.model_checkpoint.save_weights_only,callbacks.model_checkpoint.auto_insert_metric_name,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,Model
0,0.043201,0.013768,31.459143,5.232837,0.018140,0.004181,3.210601,0.537035,1640,772,1.759786e+09,{'runtime': 1640},48,0.000031,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",0.000021,0.003978,0.000021,0.004601,0.000006,0.000006,0.001691,0.000006,0.002499,26544,0.000021,0.003978,0.000021,0.004601,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",True,{'scaler_features': {'metadata': {'fit_on': 't...,32,4,False,False,True,2048,10,2048,debug,2048,True,1167436,1167436,unibo+reconstruction+residual_autoencoder_85ba...,lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,0.5,10,0.0005,0,lmetk.data.datasources.unibo.loader.UNIBO21Loader,/home/levtel/projects/TabPHM/datasets,UNIBO21,SOC,NaN,NaN,1,64,0,0,reconstruction,True,[features],"[5, 5, 5, 5, 5]","[1, 1, 1, 1, 1]",lmetk.model.autoenco

Dataset key: PRONOSTIA
  Model key: lmetk.model.autoencoder.wrapper.AutoencoderWrapper
 best_hyperparameters     model.kernels: (5, 5, 5, 5, 5)
 best_hyperparameters     model.strides: (1, 1, 1, 1, 1)
 best_hyperparameters     model.dilations: (1, 2, 4, 8, 1)
 best_hyperparameters     model.output_channels: (16, 32, 64, 64, 64)
 best_hyperparameters     run_names: ['pronostia+reconstruction+residual_autoencoder_2444eab6']
 best_performance     optimized_on: {'metric': 'val/loss', 'strategy': 'min'}
 best_performance     metrics: defaultdict(<class 'dict'>, {'loss': {'val': {'mean': np.float64(5.845259875059128e-05), 'std': np.float64(nan)}, 'test': {'mean': np.float64(5.845259875059128e-05), 'std': np.float64(nan)}}, 'mse': {'val': {'mean': np.float64(5.8452591474633664e-05), 'std': np.float64(nan)}, 'test': {'mean': np.float64(5.8452591474633664e-05), 'std': np.float64(nan)}}, 'mae': {'val': {'mean': np.float64(0.004978187382221222), 'std': np.float64(nan)}, 'test': {'mean': np.float6

Model                    model.kernels                   model.strides                 model.dilations                     model.output_channels  \
                                                                                                                                                                                                  
0  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (5, 5, 5, 5, 5)                 (1, 1, 1, 1, 1)                 (1, 2, 4, 8, 1)                      (16, 32, 64, 64, 64)   
2  lmetk.model.autoencoder.wrapper.AutoencoderWra...  (10, 7, 7, 7, 7, 7, 7, 7, 7, 7)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (16, 32, 64, 64, 64, 16, 32, 64, 64, 64)   

   val/loss       val/mse       val/mae      val/rmse     test/loss      test/mse      test/mae     test/rmse                                               run_name  
       mean std      mean std      mean std      mean std      mean std      mean std      mean std      mean std                                               list  
0  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  [pronostia+reconstruction+residual_autoencoder...  
2  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  [pronostia+reconstruction+residual_autoencoder...

    non_aggregated_df:


,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,_runtime,_step,_timestamp,_wandb,epoch,lr-Adam,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/loss_step,train/mae,train/mse,train/rmse,trainer/global_step,val/loss,val/mae,val/mse,val/rmse,loss,seed,tags,test,cache,train,logger,trainer,exp_name,evaluator,hydra_cfg,pin_memory,transforms,num_threads,num_workers,shuffle_val,shuffle_test,shuffle_train,val_batch_size,prefetch_factor,test_batch_size,experiment_group,train_batch_size,use_batch_sampler,model/params/total,model/params/trainable,run_name,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.fold,datasource.cache_dir,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.requires_training,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,model.output_channels,model.blocks_config,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,datamodule._target_,datamodule.num_workers,datamodule.shuffle_val,datamodule.shuffle_test,datamodule.shuffle_train,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.check_finite,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint.verbose,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.save_last,callbacks.model_checkpoint.save_top_k,callbacks.model_checkpoint.save_weights_only,callbacks.model_checkpoint.auto_insert_metric_name,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,Model
3,0.025387,0.004153,194.413725,6.799279,0.011248,0.002803,55.706442,1.933954,10122,5170,1.759796e+09,{'runtime': 10122},50,0.000031,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",0.000082,0.005265,0.000082,0.009032,0.000035,0.000033,0.003671,0.000035,0.005915,245900,0.000082,0.005265,0.000082,0.009032,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'pronostia+reconstruction+res...,"{'devices': [1], '_target_': 'lightning.pytorc...",pronostia/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",True,{'scaler_features': {'metadata': {'fit_on': 't...,32,4,False,False,True,2048,10,2048,debug,2048,True,183208,183208,pronostia+reconstruction+residual_autoencoder_...,lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,0.5,10,0.0005,0,lmetk.data.datasources.phmd_pronostia.Pronosti...,NaN,PRONOSTIA,rul,0.0,/home/levtel/projects/TabPHM/datasets/phmd_cache,1,64,0,0,reconstruction,True,[features],"[7, 7, 5, 5, 5]","[2, 2, 1, 1, 1]"

In [4]:
display(all_results[key][key_2]['sorted_aggregated_results'])

Model                    model.kernels                   model.strides                 model.dilations                     model.output_channels  \
                                                                                                                                                                                                  
0  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (5, 5, 5, 5, 5)                 (1, 1, 1, 1, 1)                 (1, 2, 4, 8, 1)                      (16, 32, 64, 64, 64)   
2  lmetk.model.autoencoder.wrapper.AutoencoderWra...  (10, 7, 7, 7, 7, 7, 7, 7, 7, 7)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (16, 32, 64, 64, 64, 16, 32, 64, 64, 64)   
1  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (7, 7, 5, 5, 5)                 (2, 2, 1, 1, 1)                 (1, 1, 1, 1, 1)                      (16, 32, 64, 64, 64)   

   val/loss       val/mse       val/mae      val/rmse     test/loss      test/mse      test/mae     test/rmse                                               run_name  
       mean std      mean std      mean std      mean std      mean std      mean std      mean std      mean std                                               list  
0  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  [pronostia+reconstruction+residual_autoencoder...  
2  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  [pronostia+reconstruction+residual_autoencoder...  
1  0.000082 NaN  0.000082 NaN  0.005265 NaN  0.009032 NaN  0.000082 NaN  0.000082 NaN  0.005265 NaN  0.009032 NaN  [pronostia+reconstruction+residual_autoencoder...

In [ ]:
for i in all_results[key][key_2]['non_aggregated_df']['callbacks.model_checkpoint.dirpath']:
    print(str(i))
    

/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_03-30-38/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_03-30-47/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_11-49-26/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_12-25-24/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_20-06-41/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_21-16-34/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-03_04-29-43/checkpoints/checkpoints
/home/levtel/project

In [45]:
all_results[key][key_2]['non_aggregated_df']

,_runtime,_step,_timestamp,_wandb,epoch,lr-Adam,train/loss_step,trainer/global_step,loss,seed,tags,test,cache,train,logger,trainer,exp_name,evaluator,hydra_cfg,pin_memory,transforms,num_workers,shuffle_val,shuffle_test,shuffle_train,val_batch_size,prefetch_factor,test_batch_size,experiment_group,train_batch_size,use_batch_sampler,model/params/total,model/params/trainable,run_name,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,num_threads,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.fold,datasource.cache_dir,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.requires_training,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,model.output_channels,model.blocks_config,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,datamodule._target_,datamodule.num_workers,datamodule.shuffle_val,datamodule.shuffle_test,datamodule.shuffle_train,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.check_finite,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint.verbose,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.save_last,callbacks.model_checkpoint.save_top_k,callbacks.model_checkpoint.save_weights_only,callbacks.model_checkpoint.auto_insert_metric_name,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,Model
52,29813,31573,1.759431e+09,{'runtime': 29813},20,0.00500,6.167217e-05,1573580,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'pronostia+reconstruction+res...,"{'devices': [1], '_target_': 'lightning.pytorc...",pronostia/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",True,{'scaler_features': {'metadata': {'fit_on': 't...,4,False,False,True,1024,10,1024,debug,128,True,183208,183208,pronostia+reconstruction+residual_autoencoder_...,0.015844,0.003353,1487.310064,21.117129,0.006290,0.001688,67.621402,2.877372,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",1.429764e-04,0.007379,2.859528e-04,0.011957,6.198527e-02,0.012541,1.239705e-01,2.489684e-01,1.429764e-04,0.007379,2.859528e-04,0.011957,NaN,lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,0.5,10,0.0100,0,lmetk.data.datasources.phmd_pronostia.Pronosti...,NaN,PRONOSTIA,rul,0.0,/home/levtel/projects/TabPHM/datasets/phmd_cache,1,64,0,0,reconstruction,True,[features]

In [36]:
# If run_name is the first element in the tuple
